# Demo: Live Object Detection Inference

Notebook demo cho phép user upload ảnh test và chạy inference với 4 models: YOLO11n, YOLO11s, Faster R-CNN, DETR.

In [1]:
%pip install -q ultralytics==8.3.0 transformers torch torchvision pillow matplotlib seaborn ipywidgets  numpy>=2.0.0

In [2]:
import time
import io
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from IPython.display import display, HTML

import torch
import torchvision
from torchvision import transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from ultralytics import YOLO
from transformers import DetrForObjectDetection, DetrImageProcessor

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
torch.manual_seed(42)
np.random.seed(42)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device: cuda


## 1. Tải các models

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
def resolve_assignment2_root() -> Path:
    """Tìm thư mục assignment_2"""
    try:
        from google.colab import drive
        drive_path = Path('/content/drive/MyDrive/Sem 2 2025-2026/DL-CV/CO5085_DeepLearning_CV/assignment_2')
        if drive_path.exists():
            return drive_path
    except:
        pass

    cwd = Path.cwd()
    if cwd.name == 'assignment_2':
        return cwd
    if (cwd / 'assignment_2').exists():
        return cwd / 'assignment_2'
    return cwd

ASSIGNMENT2_ROOT = resolve_assignment2_root()

YOLO11N_WEIGHT = ASSIGNMENT2_ROOT / 'yolo11n_w.pt'
YOLO11S_WEIGHT = ASSIGNMENT2_ROOT / 'yolo11s_w.pt'
FASTRCNN_WEIGHT = ASSIGNMENT2_ROOT / 'fast_rcnn_w.pth'
DETR_WEIGHT = ASSIGNMENT2_ROOT / 'detr_r50_w.pt'

print(f'ASSIGNMENT2_ROOT: {ASSIGNMENT2_ROOT}')
print(f'YOLO11N_WEIGHT exists: {YOLO11N_WEIGHT.exists()}')
print(f'YOLO11S_WEIGHT exists: {YOLO11S_WEIGHT.exists()}')
print(f'FASTRCNN_WEIGHT exists: {FASTRCNN_WEIGHT.exists()}')
print(f'DETR_WEIGHT exists: {DETR_WEIGHT.exists()}')

# Constants
NUM_CLASSES_FASTRCNN = 81
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50

COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck',
    'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench',
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra',
    'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard',
    'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon',
    'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog',
    'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table',
    'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave',
    'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors',
    'teddy bear', 'hair drier', 'toothbrush'
]

print('\nCấu hình xong')

ASSIGNMENT2_ROOT: /content/drive/MyDrive/Sem 2 2025-2026/DL-CV/CO5085_DeepLearning_CV/assignment_2
YOLO11N_WEIGHT exists: True
YOLO11S_WEIGHT exists: True
FASTRCNN_WEIGHT exists: True
DETR_WEIGHT exists: True

Cấu hình xong


In [12]:
import warnings
warnings.filterwarnings('ignore')

print("Đang load YOLO11n...")
yolo11n = YOLO(YOLO11N_WEIGHT)

print("Đang load YOLO11s...")
yolo11s = YOLO(YOLO11S_WEIGHT)

print("Đang load Faster R-CNN...")
faster_rcnn = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None, num_classes=NUM_CLASSES_FASTRCNN)
faster_rcnn.load_state_dict(torch.load(FASTRCNN_WEIGHT, map_location=device))
faster_rcnn.to(device)
faster_rcnn.eval()

print("Đang load DETR...")
detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
detr = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")

# Load weights for DETR (handling checkpoint dict and key mapping)
try:
    checkpoint = torch.load(DETR_WEIGHT, map_location=device)
    # Extract state_dict if it's a checkpoint dict
    state_dict = checkpoint.get('model_state_dict', checkpoint)

    # Fix transformers version key mismatch (e.g. model.backbone.model -> model.backbone.conv_encoder.model)
    new_state_dict = {}
    for k, v in state_dict.items():
        k = k.replace('model.backbone.model.', 'model.backbone.conv_encoder.model.')
        new_state_dict[k] = v

    detr.load_state_dict(new_state_dict, strict=False)
except Exception as e:
    print(f"Lỗi khi load DETR: {e}")

detr.to(device)
detr.eval()

print("\nĐã load thành công tất cả 4 models vào bộ nhớ!")

Đang load YOLO11n...
Đang load YOLO11s...
Đang load Faster R-CNN...
Đang load DETR...


Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Đã load thành công tất cả 4 models vào bộ nhớ!


In [14]:
to_tensor = T.ToTensor()

def infer_yolo(model, img_array: np.ndarray) -> Dict:
    """YOLO inference"""
    img_pil = Image.fromarray(img_array)
    r = model.predict(source=img_pil, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, verbose=False)[0]

    if r.boxes is None or len(r.boxes) == 0:
        return {'boxes': torch.zeros((0, 4)), 'scores': torch.zeros((0,)), 'labels': torch.zeros((0,), dtype=torch.int64)}

    return {
        'boxes': r.boxes.xyxy.cpu().float(),
        'scores': r.boxes.conf.cpu().float(),
        'labels': (r.boxes.cls.cpu().long() + 1)
    }

def infer_faster_rcnn(model, img_array: np.ndarray) -> Dict:
    """Faster R-CNN inference"""
    image = Image.fromarray(img_array)
    inp = to_tensor(image).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(inp)[0]

    keep = out['scores'].detach().cpu() >= CONF_THRESHOLD
    if keep.sum().item() == 0:
        return {'boxes': torch.zeros((0, 4)), 'scores': torch.zeros((0,)), 'labels': torch.zeros((0,), dtype=torch.int64)}

    return {
        'boxes': out['boxes'][keep].detach().cpu().float(),
        'scores': out['scores'][keep].detach().cpu().float(),
        'labels': out['labels'][keep].detach().cpu().long()
    }

def infer_detr(model, img_array: np.ndarray) -> Dict:
    """DETR inference"""
    image = Image.fromarray(img_array)
    inputs = detr_processor(images=image, return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(device)

    with torch.no_grad():
        outputs = model(pixel_values=pixel_values)

    logits = outputs.logits[0]
    boxes = outputs.pred_boxes[0]
    probs = torch.softmax(logits, dim=-1)[:, :-1]
    scores, labels = torch.max(probs, dim=-1)
    keep = scores > CONF_THRESHOLD

    w, h = image.size
    out_boxes = []
    out_scores = []
    out_labels = []

    DETR_COCO_MAP = {
        0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10,
        10: 11, 11: 13, 12: 14, 13: 15, 14: 16, 15: 17, 16: 18, 17: 19, 18: 20, 19: 21
    }

    for b, s, l in zip(boxes[keep], scores[keep], labels[keep]):
        li = int(l.item())
        if li >= len(COCO_CLASSES):
            continue

        cx, cy, bw, bh = b.detach().cpu().numpy().tolist()
        x1 = max(0.0, (cx - bw / 2.0) * w)
        y1 = max(0.0, (cy - bh / 2.0) * h)
        x2 = min(float(w), (cx + bw / 2.0) * w)
        y2 = min(float(h), (cy + bh / 2.0) * h)

        if x2 - x1 < 1 or y2 - y1 < 1:
            continue

        out_boxes.append([x1, y1, x2, y2])
        out_scores.append(float(s.item()))
        out_labels.append(li + 1)

    if len(out_boxes) == 0:
        return {'boxes': torch.zeros((0, 4)), 'scores': torch.zeros((0,)), 'labels': torch.zeros((0,), dtype=torch.int64)}

    return {
        'boxes': torch.tensor(out_boxes, dtype=torch.float32),
        'scores': torch.tensor(out_scores, dtype=torch.float32),
        'labels': torch.tensor(out_labels, dtype=torch.int64)
    }

print('✓ Inference functions defined')

✓ Inference functions defined


In [15]:
import io
import time
import math
import numpy as np
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Global state
uploaded_image = None
uploaded_image_array = None

# --- UI Elements ---
uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='1. Upload Image',
    button_style='success'
)

model_options = ['YOLO11n', 'YOLO11s', 'Faster R-CNN', 'DETR']
model_selector = widgets.SelectMultiple(
    options=model_options,
    value=['YOLO11n', 'YOLO11s', 'Faster R-CNN', 'DETR'],
    description='2. Models:',
    disabled=False,
    style={'description_width': 'initial'}
)

run_btn = widgets.Button(description="3. Run Inference", button_style='primary', icon='play')
out_display = widgets.Output()

# --- Callbacks ---
def on_upload(change):
    global uploaded_image, uploaded_image_array
    with out_display:
        clear_output()
        if not uploader.value:
            return

        try:
            uploaded_file = uploader.value[0]
            content = uploaded_file['content']
            name = uploaded_file['name']
        except KeyError:
            name = list(uploader.value.keys())[0]
            content = uploader.value[name]['content']

        image_bytes = io.BytesIO(content)
        uploaded_image = Image.open(image_bytes).convert('RGB')
        uploaded_image_array = np.array(uploaded_image)

        print(f'✓ Uploaded: {name} | Size: {uploaded_image.size}')

        preview_width = 400
        w, h = uploaded_image.size
        preview_height = int(preview_width * h / w)
        display(uploaded_image.resize((preview_width, preview_height)))
        print("\nTiếp theo: Chọn model ở bên trên và nhấn 'Run Inference'")

uploader.observe(on_upload, names='value')

def on_run_clicked(b):
    with out_display:
        clear_output()
        if uploaded_image_array is None:
            print('Vui lòng upload ảnh trước (Bước 1).')
            return

        selected_models = model_selector.value
        if not selected_models:
            print('Vui lòng chọn ít nhất 1 model (Bước 2).')
            return

        print(f'Đang chạy inference với: {", ".join(selected_models)}...')
        print('='*60)

        results = []

        def draw_detections(ax, img_array, pred, title, latency_ms):
            ax.imshow(img_array)
            boxes = pred['boxes'].numpy() if hasattr(pred['boxes'], 'numpy') else pred['boxes']
            scores = pred['scores'].numpy() if hasattr(pred['scores'], 'numpy') else pred['scores']
            labels = pred['labels'].numpy() if hasattr(pred['labels'], 'numpy') else pred['labels']

            for box, score, label in zip(boxes, scores, labels):
                x1, y1, x2, y2 = box
                w, h = x2 - x1, y2 - y1
                rect = patches.Rectangle((x1, y1), w, h, linewidth=2, edgecolor='lime', facecolor='none')
                ax.add_patch(rect)
                label_name = COCO_CLASSES[int(label) - 1] if int(label) - 1 < len(COCO_CLASSES) else f'Class {label}'
                text = f'{label_name} {score:.2f}'
                ax.text(x1, max(5, y1 - 5), text, fontsize=8, color='white',
                       bbox=dict(facecolor='lime', alpha=0.7, edgecolor='none'))
            ax.set_title(f'{title}\n{latency_ms:.1f}ms | {len(boxes)} detections', fontsize=10, fontweight='bold')
            ax.axis('off')

        try:
            if 'YOLO11n' in selected_models:
                t0 = time.time()
                pred = infer_yolo(yolo11n, uploaded_image_array)
                results.append(('YOLO11n', pred, (time.time() - t0) * 1000))

            if 'YOLO11s' in selected_models:
                t0 = time.time()
                pred = infer_yolo(yolo11s, uploaded_image_array)
                results.append(('YOLO11s', pred, (time.time() - t0) * 1000))

            if 'Faster R-CNN' in selected_models:
                t0 = time.time()
                pred = infer_faster_rcnn(faster_rcnn, uploaded_image_array)
                results.append(('Faster R-CNN', pred, (time.time() - t0) * 1000))

            if 'DETR' in selected_models:
                t0 = time.time()
                pred = infer_detr(detr, uploaded_image_array)
                results.append(('DETR', pred, (time.time() - t0) * 1000))

            n = len(results)
            cols = 2 if n > 1 else 1
            rows = math.ceil(n / cols)
            fig, axes = plt.subplots(rows, cols, figsize=(16 if cols==2 else 8, 7 * rows))

            if n == 1:
                axes = [axes]
            elif n > 1:
                axes = axes.flatten()

            for i, (title, pred, latency) in enumerate(results):
                draw_detections(axes[i], uploaded_image_array, pred, title, latency)

            for j in range(len(results), len(axes)):
                axes[j].axis('off')

            plt.tight_layout()
            plt.show()
            print('\nInference complete!')

        except NameError as e:
            print(f"\nLỗi: {e}. Vui lòng đảm bảo bạn đã chạy các cell load model ở phía trên.")

run_btn.on_click(on_run_clicked)

# --- Display ---
ui = widgets.VBox([
    widgets.HTML('<h3>Object Detection Dashboard</h3>'),
    widgets.HBox([uploader, model_selector, run_btn]),
    out_display
])
display(ui)